In [2]:
from transformers import RobertaTokenizer, DataCollatorWithPadding
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader
from tqdm import tqdm
from peft import AutoPeftModelForSequenceClassification
import torch
from datasets import load_from_disk
from sklearn.metrics import accuracy_score, confusion_matrix, precision_recall_fscore_support

/Users/ngokykhanhthu/2026/chewy/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1028 20:33:21.066000 2565 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


In [3]:
DATA_PATH = "./../data/"
ROBERTA_CONFIGS_PATHS = DATA_PATH + "roberta_configs/"
FROZEN_TEST_PATH = ROBERTA_CONFIGS_PATHS + "test_dataset.csv"
ROBERTA_WEIGHTS_PATH = ROBERTA_CONFIGS_PATHS + "peft_model"
TOKENIZER_PATH = ROBERTA_CONFIGS_PATHS + "tokenizer"
TOKENIZED_TEST_PATH = ROBERTA_CONFIGS_PATHS + "tokenized_test"
id2label = {0: 'human', 1: 'LLM-generated'}


In [4]:
inference_model = AutoPeftModelForSequenceClassification.from_pretrained(ROBERTA_WEIGHTS_PATH, id2label=id2label)

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 4817.55it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.dense.bias         | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [5]:
frozen_test = pd.read_csv(FROZEN_TEST_PATH)

In [6]:
saved_tokenizer = RobertaTokenizer.from_pretrained(TOKENIZER_PATH)
data_collator = DataCollatorWithPadding(tokenizer=saved_tokenizer, return_tensors="pt")

In [ ]:
tokenized_test_dataset = load_from_disk(TOKENIZED_TEST_PATH)

# evals

In [ ]:
def _metrics(y_true, y_pred):
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true, y_pred, labels=[0, 1], zero_division=0
    )
    return {
        "macro_f1": float(f1.mean()),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "per_class": pd.DataFrame(
            {"precision": precision, "recall": recall, "f1": f1, "support": support},
            index=pd.Index(["human", "ai"], name="class"),
        ),
        "confusion_matrix": confusion_matrix(y_true, y_pred, labels=[0, 1]),
    }


def evaluate_model(inference_model, dataset, sources, batch_size=64):
    """Return overall and per-domain metrics; matrix rows=true, columns=predicted.

    `sources` must be in the same row order as `dataset`. Metrics use fixed
    labels [0, 1], including in domains containing only one class.
    """
    eval_dataloader = DataLoader(dataset, batch_size=batch_size, collate_fn=data_collator)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    inference_model.to(device)
    inference_model.eval()
    y_true, y_pred = [], []
    for batch in tqdm(eval_dataloader):
        batch = batch.to(device)
        with torch.no_grad():
            outputs = inference_model(**batch)
        y_pred.append(outputs.logits.argmax(dim=-1))
        y_true.append(batch["labels"])

    truth = torch.cat(y_true).cpu().numpy()
    predictions = torch.cat(y_pred).cpu().numpy()
    domains = np.asarray(sources)
    if not (len(truth) == len(predictions) == len(domains)):
        raise ValueError("Dataset and sources must have the same length")

    result = _metrics(truth, predictions)
    result["per_domain"] = {
        source: _metrics(truth[domains == source], predictions[domains == source])
        for source in sorted(set(domains))
    }
    return result

In [ ]:
scores = evaluate_model(inference_model, tokenized_test_dataset, frozen_test["source"])
print(scores["macro_f1"], scores["accuracy"])
scores["per_class"]